# AMPS I · Week 13 — HW12: Péclet-number sweep
**Due:** before the Week 14 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. For every table: the mesh Péclet number, the range of $u_h$, the error away from the layer.

**Problem for Q2–Q4** (not the lecture's): $\Omega=(0,1)^2$, $\mathbf b=(1,0)$, $u=0$ on $\partial\Omega$,
$$u(x,y)=x\,g(x)\sin(\pi y),\qquad g(x)=x-\frac{e^{(x-1)/\varepsilon}-e^{-1/\varepsilon}}{1-e^{-1/\varepsilon}},\qquad f=-\varepsilon\Delta u+\mathbf b\cdot\nabla u\ \text{(UFL)}.$$
An exponential outflow layer at $x=1$; "away" means the $L^2$ error on $\{x<0.9\}$. Use $h=$ `ufl.CellDiameter` (divided by the degree for P2) in $\mathrm{Pe}_h$ and $\tau$.

| Question | Topic | Points |
|---|---|:---:|
| Q1 | The 1D difference equation, optimal diffusion, consistency | 20 |
| Q2 | Péclet sweep: Galerkin, upwind, SUPG | 40 |
| Q3 | Designing $\tau$; P2 consistency | 20 |
| Q4 | LLM debugging: an SUPG script with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

**(a) (8)** P1 on a uniform mesh of $(0,1)$ for $-\varepsilon u''+bu'=0$, $b>0$, $u(0)=0$, $u(1)=1$. Derive the difference equation at an interior node, solve it
($u_i=A+Br^i$), and show that the discrete solution oscillates if and only if $\mathrm{Pe}_h=bh/(2\varepsilon)>1$. What is $u_{N-1}$ for $\mathrm{Pe}_h\to\infty$ with $N$ even?

**(b) (6)** Replace $\varepsilon$ by $\varepsilon+\tilde\varepsilon$ in the stencil. Show that the discrete solution equals the exact one at the nodes iff
$\tilde\varepsilon=\frac{bh}{2}\big(\coth\mathrm{Pe}_h-\frac1{\mathrm{Pe}_h}\big)$. (Exact solution $u(x)=\frac{e^{bx/\varepsilon}-1}{e^{b/\varepsilon}-1}$; compare the ratios of successive differences.)

**(c) (6)** Show that SUPG is **consistent**: the exact solution $u$ satisfies the SUPG equations, hence $a_{\mathrm{SUPG}}(u-u_h,v_h)=0$. Why does dropping $-\varepsilon\Delta u_h$
from the residual not matter for P1 but destroy consistency for P2? What happens to the error rate then (state your expectation; Q3 measures it)?

> *your answers to (a)–(c)*

## Q2 — Péclet sweep: Galerkin, upwind, SUPG (40 pts)

**(a) (15)** Implement the three methods for the HW problem (P1). For $N=32$ and $\varepsilon=1,10^{-1},10^{-2},10^{-3},10^{-4},10^{-6}$ print: $\mathrm{Pe}_h$, the range of $u_h$,
the $L^2$ error on $\Omega$ and on $\{x<0.9\}$ (quadrature degree 8). Save the results for (b).

**(b) (10)** Plot the away-error against $\mathrm{Pe}_h$ (log–log), three methods. Where do the curves cross?

**(c) (10)** For $\varepsilon=10^{-4}$: $N=16,32,64,128$, away-error and global error with rates for the three methods.

**(d) (5)** Explain: for which $\mathrm{Pe}_h$ is Galerkin the best choice; why upwind converges at rate 1 away from the layer; why every method has global rate $\approx0.5$.

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) the three methods, Peclet sweep at N = 32; save to /tmp/fnx/hw12_q2.npy

from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

In [ ]:
# TODO: Q2(b) away error vs Pe_h

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) eps = 1e-4, N = 16..128, rates

from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

> *your answers to (b)–(d)*

## Q3 — Designing $\tau$; P2 consistency (20 pts)

**(a) (12)** P1, $\varepsilon=10^{-4}$, $N=16,\dots,128$: SUPG with $\tau_{\text{opt}}=\frac h{2|\mathbf b|}\xi(\mathrm{Pe}_h)$, $\tau_{\text{asym}}=\frac h{2|\mathbf b|}$, $\tau_{\text{opt}}/10$ and $10\,\tau_{\text{opt}}$.
Away-error, rates and range of $u_h$. Why are $\tau_{\text{opt}}$ and $\tau_{\text{asym}}$ almost identical here, and when would they differ?

**(b) (8)** P2, $\varepsilon=10^{-3}$: SUPG with and without $-\varepsilon\Delta u_h$ in the residual. Away-error rates; compare with your expectation from Q1(c).

In [ ]:
%%fenicsx -np 2
# TODO: Q3 tau variants (P1) and P2 with / without the Laplacian

from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2).tolist()

> *your table and answers to (a), (b)*

## Q4 — LLM debugging: an SUPG script with four bugs (20 pts)

An LLM was asked: *"DOLFINx 0.11, -np 2: solve the HW12 problem with P1 + SUPG for eps = 1e-4, N = 16, 32, 64, and print the L2 errors and rates."* It produced the cell below
(run it as given).

**(a) (12)** Find the four bugs: line, what is wrong, how it shows in the output (or why it does not), how you detected it. Use the checks of this week — the range of $u_h$,
the error away from the layer, a plot of $u_h$, a run with $\mathrm{Pe}_h<1$.

**(b) (5)** Corrected cell: away-errors and rates, range of $u_h$ (compare with Q2(c)).

**(c) (3)** Which bug is invisible in the global $L^2$ error the script prints? Which one would pass a check with $\varepsilon=1$?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
eps = 1e-4

def solve(N):
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(msh)
    b = ufl.as_vector([1.0, 0.0])
    g = x[0] - (ufl.exp((x[0] - 1) / eps) - np.exp(-1 / eps)) / (1 - np.exp(-1 / eps))
    u_ex = x[0] * g * ufl.sin(ufl.pi * x[1])
    f = -eps * ufl.div(ufl.grad(u_ex)) + ufl.dot(b, ufl.grad(u_ex))

    # SUPG parameter
    h = ufl.CellVolume(msh)
    bnorm = ufl.sqrt(ufl.dot(b, b))
    Pe = bnorm * h / (2 * eps)
    tau = h / (2 * bnorm) * (1 / ufl.tanh(Pe) - 1 / Pe)

    a = eps * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx + ufl.dot(b, ufl.grad(u)) * v * ufl.dx
    a -= tau * ufl.dot(b, ufl.grad(u)) * ufl.dot(b, ufl.grad(v)) * ufl.dx     # streamline diffusion (P1: no Laplacian term)
    L = f * v * ufl.dx

    # hyperbolic-type problem: prescribe u on the inflow boundary x = 0
    inflow = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0.0))
    bc = fem.dirichletbc(0.0, inflow, V)
    uh = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=f"supg{N}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()
    err = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - u_ex) ** 2 * ufl.dx(metadata={"quadrature_degree": 8}))), op=MPI.SUM))
    return err

errs = [solve(N) for N in (16, 32, 64)]
if comm.rank == 0:
    for N, e in zip((16, 32, 64), errs):
        print(f"N = {N:3d}: L2 error {e:.3e}")
    print("rates:", np.round(np.log2(np.array(errs[:-1]) / np.array(errs[1:])), 2))

In [ ]:
%%fenicsx -np 2
# TODO: Q4 your diagnostic copy and the corrected cell

> *your table for (a) and answers to (b), (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …